# 7. Upload MP4 videos to R2 and load annotation exports into Supabase bronze

This notebook converts each full source AVI to an H.264 MP4, uploads it to the public Cloudflare R2 `annotationbronze` bucket, and loads notebook 06 exports into the Supabase `bronze` schema. Each video URL is saved in `bronze.videos`. Videos already present in R2 at the stable `original-videos/<video_id>.mp4` key are reused and not re-uploaded; delete the R2 object to get a fresh conversion and upload.

Before running, copy `.env.example` to `.env` and set:

- `SUPABASE_DB_URL` — PostgreSQL connection URL with table-creation permissions.
- `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` — write credentials for the R2 bucket; never commit them.
- `R2_ENDPOINT_URL`, `R2_BUCKET`, and `R2_PUBLIC_URL` — the R2 S3 endpoint, bucket, and public domain.

**Rerun behaviour** is controlled by `VIDEO_FILTER`:

- `VIDEO_FILTER = None`: a full reload. All exports are validated first, then all rows are deleted from the five bronze tables (`videos`, `objects`, `events`, `event_objects`, `geometries`; the tables themselves are kept so other sessions reading them cannot block the reset), and every export is loaded, one transaction per video. Supabase then mirrors `data/processed` exactly.
- `VIDEO_FILTER = "<substring of a video name>"`: only the matching videos are reloaded. Each one's video, events, event links and objects are replaced in a single transaction; every other video is left untouched. A full name selects one video, a camera id such as `G421` a camera. Geometries, the bulk of the rows, are reloaded only where needed: an object whose box count in the export matches the rows already loaded keeps its geometries (its object row is updated in place), new objects get theirs inserted, objects no longer involved lose theirs, and an object whose count changed is rewritten. After re-annotating a video without re-running the tracking, this usually means no geometry rows at all are inserted.

Loaded videos are always replaced; nothing is skipped as already up to date. `bronze.meva_event_coverage`, written by notebook 08, is never touched.


In [ ]:
import hashlib
import json
import os
import subprocess
from collections import Counter
from threading import Lock
from time import perf_counter
from pathlib import Path
from urllib.parse import quote

import boto3
from boto3.s3.transfer import TransferConfig
from botocore.config import Config
from botocore.exceptions import ClientError, ConnectTimeoutError, ReadTimeoutError
import psycopg
from psycopg.types.json import Jsonb
from dotenv import load_dotenv

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
RAW_VIDEOS = PROJECT_ROOT / "data" / "raw" / "videos"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MP4_DIR = PROCESSED_DIR / "source_mp4"
BATCH_SIZE = 500

# Which videos to reload. None: empty the five bronze tables and reload every export, so Supabase
# mirrors data/processed exactly. A substring of the video name (a full name for one video, a camera
# id such as "G421" for a camera): delete only the matching videos' rows and reload them, leaving
# every other video untouched. Loaded videos are always replaced; nothing is skipped as up to date.
# Geometries are the slow part (hundreds of thousands of box rows), so a selected video keeps the
# box rows of every object whose box count is unchanged and rewrites only new or changed objects.
# MP4s already in R2 are reused either way; delete the R2 object to re-upload one.
#VIDEO_FILTER = "2018-03-07.11-00-01.11-05-01.school.G421"  # Pilot video. Set to None to reload the whole dataset.
VIDEO_FILTER = None

load_dotenv(PROJECT_ROOT / ".env")
SUPABASE_DB_URL = os.getenv("SUPABASE_DB_URL")
R2_ACCESS_KEY_ID = os.getenv("R2_ACCESS_KEY_ID")
R2_SECRET_ACCESS_KEY = os.getenv("R2_SECRET_ACCESS_KEY")
R2_ENDPOINT_URL = os.getenv("R2_ENDPOINT_URL")
R2_BUCKET = os.getenv("R2_BUCKET", "annotationbronze")
R2_PUBLIC_URL = os.getenv("R2_PUBLIC_URL")
if R2_ENDPOINT_URL and R2_ENDPOINT_URL.rstrip("/").endswith(f"/{R2_BUCKET}"):
    R2_ENDPOINT_URL = R2_ENDPOINT_URL.rstrip("/").removesuffix(f"/{R2_BUCKET}")

missing = [
    name
    for name, value in {
        "SUPABASE_DB_URL": SUPABASE_DB_URL,
        "R2_ACCESS_KEY_ID": R2_ACCESS_KEY_ID,
        "R2_SECRET_ACCESS_KEY": R2_SECRET_ACCESS_KEY,
        "R2_ENDPOINT_URL": R2_ENDPOINT_URL,
        "R2_PUBLIC_URL": R2_PUBLIC_URL,
    }.items()
    if not value
]
if missing:
    raise EnvironmentError(f"Set {', '.join(missing)} in {PROJECT_ROOT / '.env'} before running.")

print("=" * 72, flush=True)
print("Stage 7: uploading MP4s to R2 and loading Supabase bronze", flush=True)
print(
    f"R2 bucket: {R2_BUCKET}; "
    + (f"reloading videos matching {VIDEO_FILTER!r}" if VIDEO_FILTER else "emptying the bronze tables and reloading every video"),
    flush=True,
)
print("=" * 72, flush=True)

# A whole-dataset reload empties the bronze tables instead of dropping them. DELETE only takes row locks, so it
# is not blocked by other sessions reading these tables (DROP/TRUNCATE need an exclusive lock and
# hang behind any open transaction). Deleting child tables first avoids slow cascade checks.
CLEAR_BRONZE_SQL = """
DELETE FROM bronze.geometries;
DELETE FROM bronze.event_objects;
DELETE FROM bronze.events;
DELETE FROM bronze.objects;
DELETE FROM bronze.videos;
"""

SCHEMA_SQL = """
CREATE SCHEMA IF NOT EXISTS bronze;

CREATE TABLE IF NOT EXISTS bronze.videos (
    video_id TEXT PRIMARY KEY,
    video_name TEXT NOT NULL UNIQUE,
    camera_id TEXT,
    scene TEXT,
    capture_start_local TIMESTAMP,
    capture_end_local TIMESTAMP,
    capture_time_zone TEXT,
    duration_seconds DOUBLE PRECISION,
    frame_count INTEGER,
    fps DOUBLE PRECISION,
    frame_width INTEGER,
    frame_height INTEGER,
    source_video_path TEXT,
    storage_bucket TEXT,
    storage_path TEXT UNIQUE,
    video_url TEXT,
    source_metadata JSONB NOT NULL,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
CREATE TABLE IF NOT EXISTS bronze.objects (
    object_id TEXT PRIMARY KEY,
    source_object_id TEXT NOT NULL,
    detection_count INTEGER NOT NULL,
    first_seen_seconds DOUBLE PRECISION,
    last_seen_seconds DOUBLE PRECISION,
    label_details JSONB NOT NULL
);

CREATE TABLE IF NOT EXISTS bronze.events (
    event_id TEXT PRIMARY KEY,
    video_id TEXT NOT NULL REFERENCES bronze.videos(video_id) ON DELETE CASCADE,
    event_name TEXT,
    description TEXT,
    start_frame INTEGER,
    end_frame INTEGER,
    start_seconds DOUBLE PRECISION,
    end_seconds DOUBLE PRECISION,
    source_status TEXT NOT NULL,
    actions JSONB,
    other_actions JSONB
);
-- Tables created before the action checklist existed get the columns added here.
ALTER TABLE bronze.events ADD COLUMN IF NOT EXISTS actions JSONB;
ALTER TABLE bronze.events ADD COLUMN IF NOT EXISTS other_actions JSONB;

CREATE TABLE IF NOT EXISTS bronze.event_objects (
    event_id TEXT NOT NULL REFERENCES bronze.events(event_id) ON DELETE CASCADE,
    object_id TEXT NOT NULL REFERENCES bronze.objects(object_id) ON DELETE CASCADE,
    PRIMARY KEY (event_id, object_id)
);

CREATE TABLE IF NOT EXISTS bronze.geometries (
    geometry_id TEXT PRIMARY KEY,
    object_id TEXT NOT NULL REFERENCES bronze.objects(object_id) ON DELETE CASCADE,
    source_geometry_id INTEGER NOT NULL,
    frame_index INTEGER NOT NULL,
    timestamp_seconds DOUBLE PRECISION NOT NULL,
    label TEXT,
    confidence DOUBLE PRECISION,
    spatial_position TEXT,
    bounding_box_pixels JSONB NOT NULL
);

-- Add the unique constraint only when missing: dropping and re-adding it on every run rebuilt
-- the index over all geometries and took an exclusive lock on the largest table each time.
DO $$
BEGIN
    IF NOT EXISTS (
        SELECT 1 FROM pg_constraint
        WHERE conname = 'geometries_object_source_geometry_key'
          AND conrelid = 'bronze.geometries'::regclass
    ) THEN
        ALTER TABLE bronze.geometries ADD CONSTRAINT geometries_object_source_geometry_key
            UNIQUE (object_id, source_geometry_id);
    END IF;
END $$;

CREATE INDEX IF NOT EXISTS events_video_id_idx ON bronze.events(video_id);
CREATE INDEX IF NOT EXISTS event_objects_object_id_idx ON bronze.event_objects(object_id);
CREATE INDEX IF NOT EXISTS geometries_object_id_idx ON bronze.geometries(object_id);
"""


TIMEOUT_ERRORS = (
    TimeoutError,
    ConnectTimeoutError,
    ReadTimeoutError,
    subprocess.TimeoutExpired,
    psycopg.errors.QueryCanceled,
)


def is_timeout(error):
    """True for timeouts, including psycopg connection failures whose message mentions one."""
    if isinstance(error, TIMEOUT_ERRORS):
        return True
    return isinstance(error, psycopg.OperationalError) and "timeout" in str(error).lower()


def stable_id(*parts):
    """Return a globally stable SHA-256 identifier from namespaced values."""
    return hashlib.sha256(":".join(map(str, parts)).encode("utf-8")).hexdigest()


def batches(items, size=BATCH_SIZE):
    for start in range(0, len(items), size):
        yield items[start : start + size]


def find_source_video(video_name):
    candidates = [RAW_VIDEOS / f"{video_name}.r13.avi", RAW_VIDEOS / f"{video_name}.avi"]
    return next((path for path in candidates if path.is_file()), None)


def r2_client():
    """Return a boto3 S3 client configured for Cloudflare R2."""
    return boto3.client(
        "s3",
        endpoint_url=R2_ENDPOINT_URL,
        aws_access_key_id=R2_ACCESS_KEY_ID,
        aws_secret_access_key=R2_SECRET_ACCESS_KEY,
        region_name="auto",
        config=Config(
            signature_version="s3v4",
            connect_timeout=30,
            read_timeout=900,
            retries={"max_attempts": 5, "mode": "standard"},
        ),
    )


def r2_public_url(storage_path):
    return f"{R2_PUBLIC_URL.rstrip('/')}/{quote(storage_path, safe='/')}"


def r2_object_exists(client, storage_path):
    """Return True when a non-empty object already exists at this R2 key."""
    try:
        response = client.head_object(Bucket=R2_BUCKET, Key=storage_path)
    except ClientError as error:
        status = error.response.get("ResponseMetadata", {}).get("HTTPStatusCode")
        code = str(error.response.get("Error", {}).get("Code", ""))
        if status in {404, 403} or code in {"404", "403", "NoSuchKey", "NotFound"}:
            listed = client.list_objects_v2(Bucket=R2_BUCKET, Prefix=storage_path, MaxKeys=10)
            return any(
                obj.get("Key") == storage_path and int(obj.get("Size") or 0) > 0
                for obj in listed.get("Contents") or []
            )
        raise
    return int(response.get("ContentLength") or 0) > 0


def convert_to_mp4(source_video, output_path):
    """Create a browser-compatible H.264 MP4 from the original full AVI."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.is_file() and output_path.stat().st_mtime >= source_video.stat().st_mtime:
        print(f"  Reusing MP4: {output_path.name}", flush=True)
        return output_path
    print(f"  Converting AVI to MP4: {source_video.name}", flush=True)
    subprocess.run(
        [
            "ffmpeg", "-y", "-i", str(source_video), "-map", "0:v:0", "-map", "0:a?",
            "-c:v", "libx264", "-crf", "20", "-preset", "medium", "-c:a", "aac",
            "-movflags", "+faststart", str(output_path),
        ],
        check=True,
    )
    print(f"  MP4 ready: {output_path.name}", flush=True)
    return output_path


class UploadProgress:
    """Print multipart upload progress in 10-percent increments."""

    def __init__(self, filename, total_bytes):
        self.filename = filename
        self.total_bytes = total_bytes
        self.uploaded_bytes = 0
        self.next_percent = 10
        self.lock = Lock()

    def __call__(self, bytes_transferred):
        with self.lock:
            self.uploaded_bytes += bytes_transferred
            percent = min(100, int(self.uploaded_bytes * 100 / self.total_bytes))
            while percent >= self.next_percent:
                print(f"  R2 upload {self.filename}: {self.next_percent}%", flush=True)
                self.next_percent += 10


def upload_video_to_r2(storage_path, mp4_path, client):
    """Upload an MP4 to R2 and return its public URL."""
    file_size_mib = mp4_path.stat().st_size / 1024**2
    print(f"  Uploading {mp4_path.name} to R2 ({file_size_mib:.1f} MiB)", flush=True)
    client.upload_file(
        str(mp4_path),
        R2_BUCKET,
        storage_path,
        ExtraArgs={"ContentType": "video/mp4"},
        Callback=UploadProgress(mp4_path.name, mp4_path.stat().st_size),
        Config=TransferConfig(
            multipart_threshold=8 * 1024**2,
            multipart_chunksize=8 * 1024**2,
            max_concurrency=4,
        ),
    )
    print(f"  R2 upload complete: {storage_path}", flush=True)
    return r2_public_url(storage_path)


def load_exports(metadata_path):
    video_name = metadata_path.name.removesuffix(".metadata.json")
    export_dir = metadata_path.parent
    paths = {
        "metadata": metadata_path,
        "events": export_dir / f"{video_name}.activities.json",
        "objects": export_dir / f"{video_name}.types.json",
        "geometries": export_dir / f"{video_name}.geom.json",
    }
    missing_paths = [str(path) for path in paths.values() if not path.is_file()]
    if missing_paths:
        raise FileNotFoundError("Missing export files:\n" + "\n".join(missing_paths))
    return video_name, {name: json.loads(path.read_text(encoding="utf-8")) for name, path in paths.items()}


LOCK_TIMEOUT = "60s"  # Schema DDL needs exclusive table locks; fail fast instead of waiting forever.


def db_connect():
    """Open a Supabase connection that fails fast instead of hanging.

    TCP keepalives detect a connection the pooler or network silently dropped (otherwise psycopg
    waits forever on a dead socket), and lock_timeout turns a lock wait into an error.
    """
    connection = psycopg.connect(
        SUPABASE_DB_URL,
        connect_timeout=30,
        keepalives=1,
        keepalives_idle=30,
        keepalives_interval=10,
        keepalives_count=3,
    )
    # Set explicitly: the Supabase pooler drops startup "options", so a SET is the reliable way.
    connection.execute(f"SET lock_timeout = '{LOCK_TIMEOUT}'")
    connection.commit()
    return connection


def describe_lock_holders(cursor):
    """Return a summary of other sessions holding locks on bronze tables (open transactions)."""
    cursor.execute(
        """
        SELECT a.pid, a.state, a.application_name, now() - a.xact_start AS transaction_age,
               left(regexp_replace(a.query, '\\s+', ' ', 'g'), 100) AS query
        FROM pg_stat_activity a
        WHERE a.pid <> pg_backend_pid()
          AND a.datname = current_database()
          AND a.xact_start IS NOT NULL
          AND EXISTS (
              SELECT 1 FROM pg_locks l
              JOIN pg_class c ON c.oid = l.relation
              JOIN pg_namespace n ON n.oid = c.relnamespace
              WHERE l.pid = a.pid AND n.nspname = 'bronze'
          )
        ORDER BY a.xact_start
        """
    )
    rows = cursor.fetchall()
    if not rows:
        return "No other session currently holds locks on bronze tables."
    lines = ["Sessions holding locks on bronze tables (terminate with SELECT pg_terminate_backend(<pid>)):"]
    for pid, state, app, age, query in rows:
        lines.append(f"  pid={pid} state={state!r} app={app!r} transaction_age={age} query={query!r}")
    return "\n".join(lines)


def create_tables():
    with db_connect() as connection:
        with connection.cursor() as cursor:
            try:
                print("Ensuring the bronze schema and tables exist...", flush=True)
                cursor.execute(SCHEMA_SQL)
                if not VIDEO_FILTER:
                    print("Deleting all rows from the five bronze tables...", flush=True)
                    started = perf_counter()
                    cursor.execute(CLEAR_BRONZE_SQL)
                    print(f"  Bronze tables emptied in {perf_counter() - started:.1f}s", flush=True)
            except psycopg.errors.LockNotAvailable as error:
                connection.rollback()
                raise RuntimeError(
                    f"Could not lock the bronze tables within {LOCK_TIMEOUT}. Another session has an open "
                    "transaction on them (often a client left 'idle in transaction').\n"
                    + describe_lock_holders(cursor)
                ) from error
        connection.commit()


def existing_geometry_counts(cursor, object_ids):
    """{object_id: box rows} for the stable object ids that already have geometries loaded."""
    object_ids = list(object_ids)
    if not object_ids:
        return {}
    cursor.execute(
        "SELECT object_id, COUNT(*) FROM bronze.geometries WHERE object_id = ANY(%s::text[]) GROUP BY object_id",
        (object_ids,),
    )
    return dict(cursor.fetchall())


def delete_video_rows(cursor, video_id, object_ids, keep_object_ids=()):
    """Remove one video's previously loaded rows so it can be reloaded cleanly.

    Objects have no video_id column, so they are found through the video's events and by
    their stable IDs; geometries cascade from objects, events and links cascade from videos.
    Objects in keep_object_ids are left in place (their row is updated on insert) so that
    their unchanged geometries survive; everything else of the video goes.
    """
    keep = list(keep_object_ids)
    cursor.execute(
        "DELETE FROM bronze.objects WHERE object_id IN ("
        " SELECT eo.object_id FROM bronze.event_objects eo"
        " JOIN bronze.events e ON e.event_id = eo.event_id WHERE e.video_id = %s)"
        " AND NOT (object_id = ANY(%s::text[]))",
        (video_id, keep),
    )
    object_ids = [object_id for object_id in object_ids if object_id not in set(keep)]
    if object_ids:
        cursor.execute("DELETE FROM bronze.objects WHERE object_id = ANY(%s::text[])", (object_ids,))
    cursor.execute("DELETE FROM bronze.videos WHERE video_id = %s", (video_id,))


JSON_COLUMNS = {
    "videos": {"source_metadata"},
    "events": {"actions", "other_actions"},
    "objects": {"label_details"},
    "geometries": {"bounding_box_pixels"},
}


def insert_rows(cursor, table_name, rows, upsert_on=None):
    """Insert records into the bronze schema using the caller's transaction.

    With upsert_on set to the key column, a row whose key already exists is updated in place
    instead of failing; used for objects whose geometries were kept.
    """
    if not rows:
        return
    print(f"  Inserting {len(rows):,} rows into bronze.{table_name}...", flush=True)
    columns = list(rows[0])
    statement = (
        f"INSERT INTO bronze.{table_name} ({', '.join(columns)}) "
        f"VALUES ({', '.join('%s' for _ in columns)})"
    )
    if upsert_on:
        updates = ", ".join(f"{column} = EXCLUDED.{column}" for column in columns if column != upsert_on)
        statement += f" ON CONFLICT ({upsert_on}) DO UPDATE SET {updates}"
    values = [
        tuple(
            Jsonb(row[column]) if column in JSON_COLUMNS.get(table_name, set()) else row[column]
            for column in columns
        )
        for row in rows
    ]
    written = 0
    for batch_number, batch in enumerate(batches(values), start=1):
        cursor.executemany(statement, batch)
        written += len(batch)
        if batch_number % 5 == 0 or written == len(values):
            print(f"    {table_name}: {written:,}/{len(values):,} rows", flush=True)


def validate_exports(video_name, exports):
    """Validate all local references before replacing data in Supabase."""
    if exports["metadata"].get("video_id") != stable_id("video", video_name):
        raise ValueError(f"Unexpected video_id in metadata export for {video_name}")
    object_ids = {item["object_id"] for item in exports["objects"]}
    event_actor_ids = {
        actor["object_id"] for event in exports["events"] for actor in event["actors"]
    }
    geometry_object_ids = {item["object_id"] for item in exports["geometries"]}
    unknown_ids = (event_actor_ids | geometry_object_ids) - object_ids
    if unknown_ids:
        raise ValueError(f"{video_name} references unknown objects: {sorted(unknown_ids)}")


def sync_video(video_name, exports):
    """Upload the video if R2 lacks it, then insert the video's rows in one transaction."""
    metadata = exports["metadata"]
    video_id = metadata["video_id"]

    storage_path = f"original-videos/{video_id}.mp4"
    client = r2_client()
    if r2_object_exists(client, storage_path):
        print(f"  R2 already has {storage_path}; skipping convert and upload", flush=True)
        video_url = r2_public_url(storage_path)
    else:
        source_video = find_source_video(video_name)
        if source_video is None:
            raise FileNotFoundError(f"Original AVI not found for {video_name} in {RAW_VIDEOS}")
        mp4_path = convert_to_mp4(source_video, MP4_DIR / f"{video_name}.mp4")
        video_url = upload_video_to_r2(storage_path, mp4_path, client)

    video_row = {
        "video_id": video_id,
        "video_name": video_name,
        "camera_id": metadata.get("camera_id"),
        "scene": metadata.get("scene"),
        "capture_start_local": metadata.get("capture_start_local"),
        "capture_end_local": metadata.get("capture_end_local"),
        "capture_time_zone": metadata.get("capture_time_zone"),
        "duration_seconds": metadata.get("duration_seconds"),
        "frame_count": metadata.get("frame_count"),
        "fps": metadata.get("fps"),
        "frame_width": metadata.get("frame_width"),
        "frame_height": metadata.get("frame_height"),
        "source_video_path": metadata.get("source_video_path"),
        "storage_bucket": R2_BUCKET,
        "storage_path": storage_path,
        "video_url": video_url,
        "source_metadata": metadata,
    }

    object_id_map = {
        item["object_id"]: stable_id("object", video_id, item["object_id"])
        for item in exports["objects"]
    }
    object_rows = [
        {
            "object_id": object_id_map[item["object_id"]],
            "source_object_id": item["object_id"],
            "detection_count": item["detection_count"],
            "first_seen_seconds": item.get("first_seen_seconds"),
            "last_seen_seconds": item.get("last_seen_seconds"),
            "label_details": item["labels"],
        }
        for item in exports["objects"]
    ]

    event_rows = []
    event_object_rows = []
    for item in exports["events"]:
        actor_ids = [actor["object_id"] for actor in item["actors"]]
        event_rows.append(
            {
                "event_id": item["event_id"],
                "video_id": video_id,
                "event_name": item.get("event_name"),
                "description": item.get("description"),
                "start_frame": item.get("start_frame"),
                "end_frame": item.get("end_frame"),
                "start_seconds": item.get("start_seconds"),
                "end_seconds": item.get("end_seconds"),
                "source_status": item["source_status"],
                "actions": list(item.get("actions") or []),
                "other_actions": list(item.get("other_actions") or []),
            }
        )
        event_object_rows.extend(
            {
                "event_id": item["event_id"],
                "object_id": object_id_map[object_id],
            }
            for object_id in actor_ids
        )

    geometry_rows = []
    for item in exports["geometries"]:
        geometry_rows.append(
            {
                "geometry_id": stable_id("geometry", video_id, item["geometry_id"]),
                "object_id": object_id_map[item["object_id"]],
                "source_geometry_id": item["geometry_id"],
                "frame_index": item["frame_index"],
                "timestamp_seconds": item["timestamp_seconds"],
                "label": item.get("label"),
                "confidence": item.get("confidence"),
                "spatial_position": item.get("spatial_position"),
                "bounding_box_pixels": item["bounding_box_pixels"],
            }
        )

    # One transaction per video: old rows (when reloading selected videos) are deleted and the new
    # rows inserted all-or-nothing. A whole-dataset reload already emptied the tables up front.
    with db_connect() as connection:
        with connection.cursor() as cursor:
            kept_objects = set()
            if VIDEO_FILTER:
                # An object whose box count matches what is loaded keeps its geometries: the boxes
                # come from the tracking stage, which did not change if only the annotation did.
                new_counts = Counter(row["object_id"] for row in geometry_rows)
                old_counts = existing_geometry_counts(cursor, object_id_map.values())
                kept_objects = {object_id for object_id, n in new_counts.items() if old_counts.get(object_id) == n}
                delete_video_rows(cursor, video_id, object_id_map.values(), kept_objects)
            insert_rows(cursor, "videos", [video_row])
            insert_rows(cursor, "objects", object_rows, upsert_on="object_id" if kept_objects else None)
            insert_rows(cursor, "events", event_rows)
            insert_rows(cursor, "event_objects", event_object_rows)
            new_geometries = [row for row in geometry_rows if row["object_id"] not in kept_objects]
            kept_rows = len(geometry_rows) - len(new_geometries)
            if kept_objects:
                print(
                    f"  Geometries: {len(kept_objects)} of {len(object_rows)} objects unchanged, "
                    f"{kept_rows:,} box rows kept; {len(new_geometries):,} rows to insert",
                    flush=True,
                )
            insert_rows(cursor, "geometries", new_geometries)
        connection.commit()

    return video_id, video_url, len(event_rows), len(event_object_rows), len(object_rows), len(new_geometries), kept_rows


run_started = perf_counter()
metadata_paths = sorted(PROCESSED_DIR.glob("*/meva_export/*.metadata.json"))
if VIDEO_FILTER:
    metadata_paths = [path for path in metadata_paths if VIDEO_FILTER in path.parent.parent.name]
if not metadata_paths:
    raise FileNotFoundError(
        f"No metadata exports found under {PROCESSED_DIR}"
        + (f" matching VIDEO_FILTER={VIDEO_FILTER!r}" if VIDEO_FILTER else "")
    )

print(f"Found {len(metadata_paths)} video exports. Loading and validating...", flush=True)
video_exports = [load_exports(metadata_path) for metadata_path in metadata_paths]
# Validate every selected export before anything is deleted, so a bad file never leaves Supabase half loaded.
for index, (video_name, exports) in enumerate(video_exports, start=1):
    print(f"Validating {index}/{len(video_exports)}: {video_name}", flush=True)
    validate_exports(video_name, exports)

create_tables()
print(f"Loading {len(video_exports)} videos.", flush=True)
timed_out = []
for index, (video_name, exports) in enumerate(video_exports, start=1):
    print(f"\nProcessing {index}/{len(video_exports)}: {video_name}", flush=True)
    try:
        video_id, video_url, event_count, event_object_count, object_count, geometry_count, kept_rows = sync_video(
            video_name, exports
        )
    except Exception as error:
        if not is_timeout(error):
            raise
        print(f"  TIMEOUT on {video_name}: {error!r}; skipping to the next video.", flush=True)
        timed_out.append(video_name)
        continue
    print(
        f"Synced {video_name} ({video_id[:12]}…): {event_count} events, "
        f"{event_object_count} event-object links, {object_count} objects, "
        f"{geometry_count:,} geometries inserted"
        + (f", {kept_rows:,} kept unchanged" if kept_rows else "")
        + f"\n{video_url}"
    )

if timed_out:
    print(f"\n{len(timed_out)} video(s) skipped after timeouts and missing from Supabase; rerun with VIDEO_FILTER set to each:", flush=True)
    for video_name in timed_out:
        print(f"  - {video_name}", flush=True)

print(f"\nCompleted in {perf_counter() - run_started:.1f} seconds.", flush=True)
